## AgentCore Evaluations - online evaluation for Strands Agent

In this tutorial you will learn about to use the online evaluation from AgentCore Evaluations applied to a Strands agent.

To execute this lab you should first have created the Strands agent using the code at [00-prereqs](../../00-prereqs) folder and created your custom evaluator using the code at [01-creating-custom-evaluators](../../01-creating-custom-evaluators)

### What You'll Learn
- How to run online evaluations to a trace using the AgentCore Starter toolkit

### Tutorial Details

| Information         | Details                                                                       |
|:--------------------|:------------------------------------------------------------------------------|
| Tutorial type       | Evaluating Strands agent with online evaluators (built-in and custom)         |
| Tutorial components | Setting automated evaluation with built-in and custom evaluators              |
| Tutorial vertical   | Cross-vertical                                                                |
| Example complexity  | Easy                                                                          |
| SDK used            | Amazon Bedrock AgentCore SDK / boto3                                          |

### Online evaluation

Online evaluation enables live-traffic quality monitoring of deployed agents. Unlike on-demand evaluation which analyzes specific selected interactions, online evaluation continuously evaluates your agent's performance in production environments based on real-time traffic.

Online evaluation consists of three main components. First, **session sampling and filtering** allows you to configure specific rules to evaluate agent interactions. You can set percentage-based sampling to evaluate a portion of all sessions (for example, 10%) or define conditional filters for more targeted evaluation. Second, you can choose from **multiple evaluation methods** including creating new custom evaluators, using existing custom evaluators, or selecting from built-in evaluators. Finally, the **monitoring and analysis** capabilities let you view aggregated scores in dashboards, track quality trends over time, investigate low-scoring sessions, and analyze complete interaction flows from input to output.

With online evaluation, you configure the system to automatically monitor specific data sources—either CloudWatch log groups containing agent traces or AgentCore Runtime endpoints. The service continuously processes incoming traces based on your sampling and filtering rules, applies your chosen evaluators in real-time, and outputs detailed results to CloudWatch for analysis. This evaluation type is particularly useful for production monitoring, catching quality regressions early, identifying patterns in user interactions, and maintaining consistent agent performance at scale.

Once you create and enable an online evaluation configuration, the service runs continuously in the background, evaluating sessions as they occur and providing ongoing visibility into your agent's quality metrics. You can pause, modify, or delete configurations at any time to adapt your evaluation strategy as your needs evolve.

### Generating traces on AgentCore Observability from an agent

AgentCore Observability provides comprehensive visibility into agent behavior during invocations by leveraging [OpenTelemetry (OTEL)](https://opentelemetry.io/) traces as the foundation for capturing and structuring detailed execution data. AgentCore relies on [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/) to instrument different types of OTEL traces across various agent frameworks.

When your agent is hosted on AgentCore Runtime (like our agent in this tutorial), the AgentCore Observability instrumentation is automatic, with minimal configuration. All you need to do is include `aws-opentelemetry-distro` in `requirements.txt` and AgentCore Runtime handles OTEL configuration automatically. When your agent is not running in AgentCore Runtime, you will need to instrument it with ADOT to have it available in AgentCore Observability. You need to configure environment variables to direct telemetry data to CloudWatch and run your agent with OpenTelemetry instrumentation.

The process looks as following:

![session_traces](../../images/observability_traces.png)

Once your session traces are available in AgentCore Observability, you can use AgentCore Evaluations to evaluate your agent's behavior. For online evaluations, you don't need to do anything extra. Just monitor your agent's performance from the live dashboards.

### How online evaluation works with the traces

On the online evaluation, your agent is invoked and generates traces in AgentCore Observability. Those traces are mapped to sessions and their logs are made available in Amazon CloudWatch Log groups. With the online evaluation, a developer creates an online evaluation configuration for a certain agent and defines a sample rate and the evaluators to be applied for this configuration. AgentCore Evaluations will then automatically evaluate the agent in production, analyzing the produced traces according to the set sampling rate. The developer can then use the AgentCore Observability dashboards to visualize the traces and evaluation scores from the agent to continuously update the agent according to the evaluations results.


![session_traces](../../images/online_evaluations.png)

### Retrieving information from previous tutorials

For this tutorial, we will use the Strands agent deployed in AgentCore Runtime during our prerequisites tutorial. We will evaluate it with pre-built metrics and with the `response_quality` metric we created in the `01-creating-custom-metrics` tutorial. Let's retrieve our agent and evaluator informations.

In [1]:
%store -r agent_id_strands
%store -r agent_arn_strands
%store -r evaluator_id
try:
    print("Agent Id:", agent_id_strands)
    print("Agent ARN:", agent_arn_strands)
except NameError:
    raise Exception(
        """Missing agent info from your Strands agent. Please run 00-prereqs before executing this lab"""
    )

try:
    print("Evaluator id:", evaluator_id)
except NameError:
    raise Exception(
        """Missing custom evaluator id. Please run 01-creating-custom-evaluators before executing this lab"""
    )

Agent Id: acevalstrands2-xKJy20HJDc
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:849138760372:runtime/acevalstrands2-xKJy20HJDc
Evaluator id: response_quality_for_scope_b40a1c32-vBGSJi4Nz4


### Initiating the AgentCore Evaluations client

Now let's initiate the boto3 client for the AgentCore control plane. We use `bedrock-agentcore-control` to create and manage online evaluation configurations.

An **online evaluation configuration** attaches a set of evaluators to an AgentCore Runtime along with a sampling rate. Once active, AgentCore automatically scores every sampled session as it completes — no explicit evaluation API call is needed per session. Scores are written to CloudWatch Logs and can be queried through the AgentCore evaluations API. The configuration can be paused and resumed at any time without modifying the agent.

> **Learn more:** [Configure online evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/online-evaluation.html)


In [2]:
import boto3
import json
import time
from boto3.session import Session
from IPython.display import Markdown, display

In [3]:
boto_session = Session()
region = boto_session.region_name
print(region)

us-east-1


In [4]:
cp = boto3.client("bedrock-agentcore-control", region_name=region)
iam = boto3.client("iam", region_name=region)
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

### Setting online evaluation configuration

Let's now set the online evaluation configuration. In this case, we will evaluate every trace produced as we are only using our agent for demonstration purposes. In real-life applications, you want to set the sample rate accordingly to your agent's utilization.

We will create an evaluation configuration with the 5 metrics we explored in the on-demand evaluation:
* Builtin.GoalSuccessRate
* Builtin.Correctness
* Builtin.ToolParameterAccuracy
* Builtin.ToolSelectionAccuracy and
* our custom metric: response_Quality

In [5]:
import uuid as _uuid

# Derive CloudWatch log group and OTel service name from agent ID
cw_log_group = f"/aws/bedrock-agentcore/runtimes/{agent_id_strands}-DEFAULT"
_runtime_name = agent_id_strands.rsplit("-", 1)[0]
otel_service_name = f"{_runtime_name}.DEFAULT"

# Create IAM role for the online evaluation service to assume
ONLINE_EVAL_ROLE_NAME = "AgentCoreOnlineEvaluationRole"
ONLINE_EVAL_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ONLINE_EVAL_ROLE_NAME}"

trust_policy = json.dumps(
    {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }
)
inline_policy = json.dumps(
    {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "logs:FilterLogEvents",
                    "logs:GetLogEvents",
                    "logs:DescribeLogGroups",
                    "logs:DescribeLogStreams",
                    "logs:StartQuery",
                    "logs:GetQueryResults",
                    "logs:StopQuery",
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                ],
                "Resource": "*",
            },
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream",
                ],
                "Resource": "*",
            },
        ],
    }
)
try:
    iam.create_role(
        RoleName=ONLINE_EVAL_ROLE_NAME,
        AssumeRolePolicyDocument=trust_policy,
        Description="IAM role for AgentCore online evaluation",
    )
    iam.put_role_policy(
        RoleName=ONLINE_EVAL_ROLE_NAME,
        PolicyName="AgentCoreOnlineEvalPolicy",
        PolicyDocument=inline_policy,
    )
    print(f"Created IAM role: {ONLINE_EVAL_ROLE_ARN}")
    time.sleep(10)  # allow IAM propagation
except iam.exceptions.EntityAlreadyExistsException:
    # Update the existing role's inline policy to ensure Bedrock permissions are present
    iam.put_role_policy(
        RoleName=ONLINE_EVAL_ROLE_NAME,
        PolicyName="AgentCoreOnlineEvalPolicy",
        PolicyDocument=inline_policy,
    )
    print(f"Updated existing IAM role: {ONLINE_EVAL_ROLE_ARN}")
    time.sleep(5)

# Unique name to avoid ConflictException on re-runs (name must be alphanumeric+underscores only)
_config_name = f"strands_agent_eval2_{_uuid.uuid4().hex[:8]}"

# Create the online evaluation configuration
response = cp.create_online_evaluation_config(
    onlineEvaluationConfigName=_config_name,
    rule={"samplingConfig": {"samplingPercentage": 100.0}},
    dataSourceConfig={
        "cloudWatchLogs": {
            "logGroupNames": [cw_log_group],
            "serviceNames": [otel_service_name],
        }
    },
    evaluators=[
        {"evaluatorId": "Builtin.GoalSuccessRate"},
        {"evaluatorId": "Builtin.Correctness"},
        {"evaluatorId": "Builtin.ToolParameterAccuracy"},
        {"evaluatorId": "Builtin.ToolSelectionAccuracy"},
        {"evaluatorId": evaluator_id},
    ],
    evaluationExecutionRoleArn=ONLINE_EVAL_ROLE_ARN,
    enableOnCreate=True,
)
print(f"Created online eval config: {_config_name}")

Updated existing IAM role: arn:aws:iam::849138760372:role/AgentCoreOnlineEvaluationRole


Created online eval config: strands_agent_eval2_ebdc095d


### Analyzing the evaluation configuration

Let's see the configuration ID from our online evaluation configuration:

In [6]:
print("Online Evaluation Configuration Id:", response["onlineEvaluationConfigId"])

Online Evaluation Configuration Id: strands_agent_eval2_ebdc095d-S0r07tDoxA


We can also see the details of the configuration created to confirm it is already enabled:

In [7]:
cp.get_online_evaluation_config(
    onlineEvaluationConfigId=response["onlineEvaluationConfigId"]
)

{'ResponseMetadata': {'RequestId': '02869f36-fa5d-48a7-825e-52a40fb79838',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 05 May 2026 21:09:14 GMT',
   'content-type': 'application/json',
   'content-length': '1085',
   'connection': 'keep-alive',
   'x-amzn-requestid': '02869f36-fa5d-48a7-825e-52a40fb79838',
   'x-amz-apigw-id': 'c6NjQETrIAMEWzg=',
   'x-amzn-trace-id': 'Root=1-69fa5c7a-23aedd971277b6744d59debf'},
  'RetryAttempts': 0},
 'onlineEvaluationConfigArn': 'arn:aws:bedrock-agentcore:us-east-1:849138760372:online-evaluation-config/strands_agent_eval2_ebdc095d-S0r07tDoxA',
 'onlineEvaluationConfigId': 'strands_agent_eval2_ebdc095d-S0r07tDoxA',
 'onlineEvaluationConfigName': 'strands_agent_eval2_ebdc095d',
 'rule': {'samplingConfig': {'samplingPercentage': 100.0}},
 'dataSourceConfig': {'cloudWatchLogs': {'logGroupNames': ['/aws/bedrock-agentcore/runtimes/acevalstrands2-xKJy20HJDc-DEFAULT'],
   'serviceNames': ['acevalstrands2.DEFAULT']}},
 'evaluators': [{'evaluatorI

### Invoking agent to trigger evaluation

Let's now invoke our agent with a couple new queries to trigger our online evaluation. This time we will invoke our agent with boto3 as once the endpoint is available you can invoke it from any interface.

In [8]:
import boto3

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)


def invoke_agent_runtime(agent_arn, prompt):
    boto3_response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": prompt}),
    )
    if "text/event-stream" in boto3_response.get("contentType", ""):
        content = []
        for line in boto3_response["response"].iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    line = line[6:]
                    print(line)
                    content.append(line)
        display(Markdown("\n".join(content)))
    else:
        try:
            events = []
            for event in boto3_response.get("response", []):
                events.append(event)
        except Exception as e:
            events = [f"Error reading EventStream: {e}"]
        display(Markdown(json.loads(events[0].decode("utf-8"))))
    return boto3_response

In [9]:
response = invoke_agent_runtime(agent_arn_strands, "How much is 7+9+10*2?")

The answer is **36**.

Here's how it breaks down:
- 10 × 2 = 20 (multiplication first)
- 7 + 9 + 20 = 36 (then addition, left to right)

In [10]:
response = invoke_agent_runtime(agent_arn_strands, "Is it raining?")

I'll check the current weather for you.No, it's not raining. The current weather is **sunny**! ☀️

In [11]:
response = invoke_agent_runtime(agent_arn_strands, "how much is 20% of 300?")

20% of 300 is **60**.

In [12]:
response = invoke_agent_runtime(agent_arn_strands, "What can you do?")

I can help you with several things:

1. **Math Calculations**: I can perform a wide range of mathematical operations, including:
   - Basic arithmetic (addition, subtraction, multiplication, division)
   - Algebraic expressions and equation solving
   - Calculus operations (derivatives, integrals, limits)
   - Series expansions
   - Matrix operations
   - Trigonometric and other mathematical functions

2. **Weather Information**: I can tell you the current weather conditions.

3. **General Assistance**: I can answer questions and help you with various tasks.

Feel free to ask me to:
- Calculate something like "What is 25 * 4?" or "Solve x² + 2x - 8 = 0"
- Find a derivative or integral
- Check the current weather
- Help with other questions you might have

What would you like help with?

In [13]:
response = invoke_agent_runtime(agent_arn_strands, "What is the capital of NY State?")

The capital of New York State is **Albany**. 

Albany has been the state capital since 1797 and is located in the eastern part of New York State along the Hudson River. It's home to the New York State Capitol building and serves as the center of state government.

### Visualizing Online Evaluation

Once you create enought interactions with your agent you can use the [AgentCore Observability console ](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents) to visualize how it is performing according to your online evaluation configuration. 

Navigate to your agent `DEFAULT` endpoint to see the current evaluations

**Important**: The evaluation results might take a while to appear in your dashboard. If you evaluation dashboard is empty, please wait a couple of minutes to check it again.

Once available you will be able to see your metrics directly in the agent's traces:
![image.png](../../images/online_evaluations_dashboard.png)

### Congratulations!

You have created your first Online Evaluation Configuration! You can now create custom metrics and evaluate your agent on-demand and online with AgentCore Evaluations!